In [ ]:
import duckdb

con = duckdb.connect(database='football_data.db')

In [ ]:
con.execute('SHOW TABLES;').fetchdf()

In [ ]:
con.execute("select * from ratings").fetch_df()

In [ ]:
query = """
CREATE VIEW IF NOT EXISTS interleague_factors AS
WITH prev_season AS (
    SELECT
        team,
        country,
        season_end_year AS prev_season,
        league AS prev_league,
        home_g / home_p AS prev_home_gpg,
        home_xg / home_p AS prev_home_xgpg,
        home_ga / home_p AS prev_home_gapg,
        home_xga / home_p AS prev_home_xgapg,
        away_g / away_p AS prev_away_gpg,
        away_xg / away_p AS prev_away_xgpg,
        away_ga / away_p AS prev_away_gapg,
        away_xga / away_p AS prev_away_xgapg,
        total_g / total_p AS prev_total_gpg,
        total_xg / total_p AS prev_total_xgpg,
        total_ga / total_p AS prev_total_gapg,
        total_xga / total_p AS prev_total_xgapg
    FROM ratings
    ),
    curr_season AS (
        SELECT
            team,
            country,
            tier,
            prev_tier,
            season_end_year,
            league,
            home_g / home_p AS home_gpg,
            home_xg / home_p AS home_xgpg,
            home_ga / home_p AS home_gapg,
            home_xga / home_p AS home_xgapg,
            away_g / away_p AS away_gpg,
            away_xg / away_p AS away_xgpg,
            away_ga / away_p AS away_gapg,
            away_xga / away_p AS away_xgapg,
            total_g / total_p AS total_gpg,
            total_xg / total_p AS total_xgpg,
            total_ga / total_p AS total_gapg,
            total_xga / total_p AS total_xgapg
        FROM ratings
    ),
joined AS (
SELECT
    cs.season_end_year AS current_season,
    cs.team,
    cs.country,
    cs.tier,
    cs.prev_tier,
    cs.league,
    cs.home_gpg,
    cs.home_xgpg,
    cs.home_gapg,
    cs.home_xgapg,
    cs.away_gpg,
    cs.away_xgpg,
    cs.away_xgpg,
    cs.away_gapg,
    cs.away_xgapg,
    cs.total_gpg,
    cs.total_xgpg,
    cs.total_gapg,
    cs.total_xgapg,
    ps.prev_home_gpg,
    ps.prev_home_xgpg,
    ps.prev_home_gapg,
    ps.prev_home_xgapg,
    ps.prev_away_gpg,
    ps.prev_away_xgpg,
    ps.prev_away_gapg,
    ps.prev_away_xgapg,
    ps.prev_total_gpg,
    ps.prev_total_xgpg,
    ps.prev_total_gapg,
    ps.prev_total_xgapg
FROM curr_season cs
LEFT JOIN prev_season ps
ON cs.team = ps.team AND cs.season_end_year = ps.prev_season + 1
)
SELECT 
country,
tier,
prev_tier,
league,
mean(sf_home_gpg) AS league_sf_home_gpg,
mean(sf_home_xgpg) AS league_sf_home_xgpg,
mean(sf_home_gapg) AS league_sf_home_gapg,
mean(sf_home_xgapg) AS league_sf_home_xgapg,
mean(sf_away_gpg) AS league_sf_away_gpg,
mean(sf_away_xgpg) AS league_sf_away_xgpg,
mean(sf_away_gapg) AS league_sf_away_gapg,
mean(sf_away_xgapg) AS league_sf_away_xgapg,
mean(sf_total_gpg) AS league_sf_total_gpg,
mean(sf_total_xgpg) AS league_sf_total_xgpg,
mean(sf_total_gapg) AS league_sf_total_gapg,
mean(sf_total_xgapg) AS league_sf_total_xgapg
FROM(
SELECT
    current_season,
    country,
    tier,
    prev_tier,
    league,
    team,
    home_gpg / prev_home_gpg AS sf_home_gpg,
    home_xgpg / prev_home_xgpg AS sf_home_xgpg,
    home_gapg / prev_home_gapg AS sf_home_gapg,
    home_xgapg / prev_home_xgapg AS sf_home_xgapg,
    away_gpg / prev_away_gpg AS sf_away_gpg,
    away_xgpg / prev_away_xgpg AS sf_away_xgpg,
    away_gapg / prev_away_gapg AS sf_away_gapg,
    away_xgapg / prev_away_xgapg AS sf_away_xgapg,
    total_gpg / prev_total_gpg AS sf_total_gpg,
    total_xgpg / prev_total_xgpg AS sf_total_xgpg,
    total_gapg / prev_total_gapg AS sf_total_gapg,
    total_xgapg / prev_total_xgapg AS sf_total_xgapg
FROM joined
WHERE prev_tier IS NOT NULL
)
GROUP BY country, tier, prev_tier, league
ORDER BY country, tier, prev_tier;
"""
df = con.execute(query)

In [ ]:
con.execute("SELECT * from interleague_factors").fetch_df()

In [ ]:
con.close()

In [ ]:
query = """
WITH promoted_teams AS (
    SELECT *
    FROM ratings
    WHERE season_start_flag = 'promoted'
),
previous_season_stats AS (
    SELECT 
        team,
        season_end_year,
        country,
        tier,
        league,
        home_g,
        home_ga,
        home_xg,
        home_xga,
        home_pts,
        home_p,
        away_g,
        away_ga,
        away_xg,
        away_xga,
        away_pts,
        away_p,
        total_g,
        total_ga,
        total_xg,
        total_xga,
        total_pts,
        total_p,
        finishing_position
    FROM ratings
)
    SELECT
        p.season_end_year AS current_season,
        p.team,
        p.country,
        p.tier AS current_tier,
        p.league AS current_league,
    p.prev_tier,
    
    -- Current season stats
    p.home_g AS curr_home_g,
    p.home_ga AS curr_home_ga,
    p.home_xg AS curr_home_xg,
    p.home_xga AS curr_home_xga,
    p.home_pts AS curr_home_pts,
    p.home_p AS curr_home_p,
    p.away_g AS curr_away_g,
    p.away_ga AS curr_away_ga,
    p.away_xg AS curr_away_xg,
    p.away_xga AS curr_away_xga,
    p.away_pts AS curr_away_pts,
    p.away_p AS curr_away_p,
    p.total_g AS curr_total_g,
    p.total_ga AS curr_total_ga,
    p.total_xg AS curr_total_xg,
    p.total_xga AS curr_total_xga,
    p.total_pts AS curr_total_pts,
    p.total_p AS curr_total_p,
    p.finishing_position AS curr_finishing_position,
    
    -- Previous season stats
    prev.season_end_year AS prev_season,
    prev.tier AS prev_tier_actual,
    prev.league AS prev_league,
    prev.home_g AS prev_home_g,
    prev.home_ga AS prev_home_ga,
    prev.home_xg AS prev_home_xg,
    prev.home_xga AS prev_home_xga,
    prev.home_pts AS prev_home_pts,
    prev.home_p AS prev_home_p,
    prev.away_g AS prev_away_g,
    prev.away_ga AS prev_away_ga,
    prev.away_xg AS prev_away_xg,
    prev.away_xga AS prev_away_xga,
    prev.away_pts AS prev_away_pts,
    prev.away_p AS prev_away_p,
    prev.total_g AS prev_total_g,
    prev.total_ga AS prev_total_ga,
    prev.total_xg AS prev_total_xg,
    prev.total_xga AS prev_total_xga,
    prev.total_pts AS prev_total_pts,
    prev.total_p AS prev_total_p,
    prev.finishing_position AS prev_finishing_position
FROM promoted_teams p
LEFT JOIN previous_season_stats prev 
    ON p.team = prev.team 
    AND p.season_end_year - 1 = prev.season_end_year
ORDER BY p.season_end_year DESC, p.team;
"""
df = con.execute(query).fetchdf()

In [ ]:
gdiff_df = df[['current_season', 'team', 'current_league', 'curr_home_g', 'curr_away_g', 'curr_total_g', 'curr_total_p', 'prev_home_g', 'prev_away_g', 'prev_total_g', 'prev_total_p']][(df['current_season'] != 2026) & (df['curr_total_p'] >2 ) & df['prev_total_g'].notnull()]

gdiff_df['curr_total_gpg'] = gdiff_df['curr_total_g'] / gdiff_df['curr_total_p']
gdiff_df['prev_total_gpg'] = gdiff_df['prev_total_g'] / gdiff_df['prev_total_p']

In [ ]:
diff_df = df[['current_season', 'team', 'current_league', 'curr_total_g', 'curr_total_ga', 'curr_total_p', 'prev_total_g', 'prev_total_ga', 'prev_total_p']][(df['current_season'] != 2026) & (df['curr_total_p'] >2 ) & df['prev_total_g'].notnull()]

diff_df['curr_total_gpg'] = diff_df['curr_total_g'] / diff_df['curr_total_p']
diff_df['prev_total_gpg'] = diff_df['prev_total_g'] / diff_df['prev_total_p']
diff_df['curr_total_gapg'] = diff_df['curr_total_ga'] / diff_df['curr_total_p']
diff_df['prev_total_gapg'] = diff_df['prev_total_ga'] / diff_df['prev_total_p']
diff_df['shrink_factor_gpg'] = diff_df['curr_total_gpg'] / diff_df['prev_total_gpg']
diff_df['shrink_factor_gapg'] = diff_df['curr_total_gapg'] / diff_df['prev_total_gapg']
diff_df.groupby(['current_league'])[['shrink_factor_gpg', 'shrink_factor_gapg']].mean().reset_index().sort_values(by='shrink_factor_gpg', ascending=False)

In [ ]:
diff_df = df[['current_season', 'team', 'current_league', 'curr_home_g', 'curr_home_ga', 'curr_home_p', 'prev_home_g', 'prev_home_ga', 'prev_home_p']][(df['current_season'] != 2026) & (df['curr_home_p'] >2 ) & df['prev_home_g'].notnull()]

diff_df['curr_home_gpg'] = diff_df['curr_home_g'] / diff_df['curr_home_p']
diff_df['prev_home_gpg'] = diff_df['prev_home_g'] / diff_df['prev_home_p']
diff_df['curr_home_gapg'] = diff_df['curr_home_ga'] / diff_df['curr_home_p']
diff_df['prev_home_gapg'] = diff_df['prev_home_ga'] / diff_df['prev_home_p']
diff_df['shrink_factor_gpg'] = diff_df['curr_home_gpg'] / diff_df['prev_home_gpg']
diff_df['shrink_factor_gapg'] = diff_df['curr_home_gapg'] / diff_df['prev_home_gapg']
diff_df.groupby(['current_league'])[['shrink_factor_gpg', 'shrink_factor_gapg']].mean().reset_index().sort_values(by='shrink_factor_gpg', ascending=False)

In [ ]:
diff_df = df[['current_season', 'team', 'current_league', 'curr_away_g', 'curr_away_ga', 'curr_away_p', 'prev_away_g', 'prev_away_ga', 'prev_away_p']][(df['current_season'] != 2026) & (df['curr_away_p'] >2 ) & df['prev_away_g'].notnull()]

diff_df['curr_away_gpg'] = diff_df['curr_away_g'] / diff_df['curr_away_p']
diff_df['prev_away_gpg'] = diff_df['prev_away_g'] / diff_df['prev_away_p']
diff_df['curr_away_gapg'] = diff_df['curr_away_ga'] / diff_df['curr_away_p']
diff_df['prev_away_gapg'] = diff_df['prev_away_ga'] / diff_df['prev_away_p']
diff_df['shrink_factor_gpg'] = diff_df['curr_away_gpg'] / diff_df['prev_away_gpg']
diff_df['shrink_factor_gapg'] = diff_df['curr_away_gapg'] / diff_df['prev_away_gapg']
diff_df.groupby(['current_league'])[['shrink_factor_gpg', 'shrink_factor_gapg']].mean().reset_index().sort_values(by='shrink_factor_gpg', ascending=False)

In [ ]:
import numpy as np

import matplotlib.pyplot as plt
def add_best_fit_line(ax, x, y):
    # Remove NaNs
    mask = ~np.isnan(x) & ~np.isnan(y)
    x_clean = x[mask]
    y_clean = y[mask]
    if len(x_clean) < 2:
        return
    # Fit line
    coeffs = np.polyfit(x_clean, y_clean, 1)
    m, b = coeffs
    x_vals = np.array([x_clean.min(), x_clean.max()])
    y_vals = m * x_vals + b
    ax.plot(x_vals, y_vals, color='red', linewidth=2, label='Best fit')
    # Display formula
    formula = f'y = {m:.2f}x + {b:.2f}'
    ax.text(0.05, 0.95, formula, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', color='red', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

def plot_prev_vs_curr_goals(gdiff_df, colname='total_g'):
    prev_col = f'prev_{colname}'
    curr_col = f'curr_{colname}'

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(
        gdiff_df[prev_col],
        gdiff_df[curr_col],
        c=gdiff_df['current_league'].astype('category').cat.codes,
        cmap='tab10',
        alpha=0.7
    )
    plt.xlabel(f'Previous Season {colname.replace("_", " ").title()}')
    plt.ylabel(f'Current Season {colname.replace("_", " ").title()}')
    plt.title(f'Current vs Previous Season {colname.replace("_", " ").title()} by League')

    handles, _ = scatter.legend_elements(prop="colors")
    leagues = gdiff_df['current_league'].astype('category').cat.categories
    plt.legend(handles, leagues, title="Current League", bbox_to_anchor=(1.05, 1), loc='upper left')

    ax = plt.gca()
    add_best_fit_line(ax, gdiff_df[prev_col].values, gdiff_df[curr_col].values)

    plt.tight_layout()
    plt.show()

def plot_prev_vs_curr_goals_by_league(gdiff_df, colname='total_g'):
    leagues = gdiff_df['current_league'].unique()
    n_leagues = len(leagues)

    prev_col = f'prev_{colname}'
    curr_col = f'curr_{colname}'

    fig, axes = plt.subplots(1, n_leagues, figsize=(5 * n_leagues, 5), sharey=True)
    if n_leagues == 1:
        axes = [axes]

    for ax, league in zip(axes, leagues):
        league_df = gdiff_df[gdiff_df['current_league'] == league]
        ax.scatter(
            league_df[prev_col],
            league_df[curr_col],
            alpha=0.7
        )
        ax.set_title(league)
        ax.set_xlabel(f'Previous Season {colname.replace("_", " ").title()}')
        ax.set_ylabel(f'Current Season {colname.replace("_", " ").title()}')
        add_best_fit_line(ax, league_df[prev_col].values, league_df[curr_col].values)

    plt.tight_layout()
    plt.show()



In [ ]:
sf = diff_df.groupby(['current_league'])[['shrink_factor_gpg', 'shrink_factor_gapg']].mean().reset_index().sort_values(by='shrink_factor_gpg', ascending=False)

diff_df['prev_total_gpg_est'] = diff_df['prev_total_gpg'] * sf.set_index('current_league').loc[diff_df['current_league'], 'shrink_factor_gpg'].values
diff_df['curr_total_gpg_est'] = diff_df['curr_total_gpg']

In [ ]:
diff_df

In [ ]:
col = 'total_gpg_est'
plot_prev_vs_curr_goals(diff_df, colname=col)

plot_prev_vs_curr_goals_by_league(diff_df, colname=col)

In [ ]:
xgdiff_df = df[['current_season', 'team', 'current_league', 'curr_home_xg', 'curr_away_xg', 'curr_total_xg', 'curr_total_p', 'prev_home_xg', 'prev_away_xg', 'prev_total_xg', 'prev_total_p']][(df['current_season'] != 2026) & (df['curr_total_p'] >2 ) & df['prev_total_xg'].notnull()]

xgdiff_df['curr_total_xgpg'] = xgdiff_df['curr_total_xg'] / xgdiff_df['curr_total_p']
xgdiff_df['prev_total_xgpg'] = xgdiff_df['prev_total_xg'] / xgdiff_df['prev_total_p']

In [ ]:
col = 'total_gpg'
plot_prev_vs_curr_goals(diff_df, colname=col)

plot_prev_vs_curr_goals_by_league(diff_df, colname=col)

In [ ]:
gdiff_df['shrink_factor'] = gdiff_df['curr_total_gpg'] / gdiff_df['prev_total_gpg']
league_sf = gdiff_df.groupby(['current_league'])['shrink_factor'].mean().reset_index().sort_values(by='shrink_factor', ascending=False)
league_sf 

In [ ]:
diff_df[gdiff_df['current_league'] == 'La_Liga']

In [ ]:
con.execute(f"DROP VIEW IF EXISTS ratings;")

In [ ]:
con.close()

In [ ]:
con.execute("""
WITH home_stats AS (
    SELECT
        season_end_year,
        country,
        tier,
        league,
        home_team AS team, 
        SUM(home_g) AS home_g,
        SUM(away_g) AS home_ga,
        SUM(home_xg) AS home_xg,
        SUM(away_xg) AS home_xga,
        SUM(
            CASE
                WHEN match_id LIKE '%playoffs%' THEN 0
                WHEN home_g > away_g THEN 3
                WHEN home_g < away_g THEN 0
                WHEN home_g = away_g THEN 1
                ELSE 0
            END
        ) AS home_pts,
        SUM(
            CASE
                WHEN status = 'played' THEN 1
                ELSE 0
            END
        ) AS home_p
    FROM matches
    WHERE match_id NOT LIKE '%playoffs%'
    GROUP BY season_end_year, country, tier, league, home_team
),
away_stats AS (
    SELECT
        season_end_year,
        country,
        tier,
        league,
        away_team AS team, 
        SUM(away_g) AS away_g,
        SUM(home_g) AS away_ga,
        SUM(away_xg) AS away_xg,
        SUM(home_xg) AS away_xga,
        SUM(
            CASE
                 WHEN match_id LIKE '%playoffs%' THEN 0
                WHEN away_g > home_g THEN 3
                WHEN away_g < home_g THEN 0
                WHEN away_g = home_g THEN 1
                ELSE 0
            END
        ) AS away_pts,
        SUM(
            CASE
                WHEN status = 'played' THEN 1
                ELSE 0
            END
        ) AS away_p
    FROM matches
    WHERE match_id NOT LIKE '%playoffs%'
    GROUP BY season_end_year, country, tier, league, away_team
),
combined_stats AS (
    SELECT
        home_stats.season_end_year,
        home_stats.country,
        home_stats.tier,
        LAG(home_stats.tier) OVER (PARTITION BY home_stats.team ORDER BY home_stats.season_end_year) AS prev_tier,
        LEAD(home_stats.tier) OVER (PARTITION BY home_stats.team ORDER BY home_stats.season_end_year) AS next_tier,
        home_stats.league,
        home_stats.team,
        home_stats.home_g,
        home_stats.home_ga,
        home_stats.home_xg,
        home_stats.home_xga,
        home_stats.home_pts,
        home_stats.home_p,
        away_stats.away_g,
        away_stats.away_ga,
        away_stats.away_xg,
        away_stats.away_xga,
        away_stats.away_pts,
        away_stats.away_p
    FROM home_stats
    JOIN away_stats USING (season_end_year, country, tier, league, team)
)
SELECT
    season_end_year,
    country,
    tier,
    prev_tier,
    next_tier,
    league,
    team,
    home_g,
    home_ga,
    home_xg,
    home_xga,
    home_pts,
    home_p,
    away_g,
    away_ga,
    away_xg,
    away_xga,
    away_pts,
    away_p,
    (home_g + away_g) AS total_g,
    (home_ga + away_ga) AS total_ga,
    (home_xg + away_xg) AS total_xg,
    (home_xga + away_xga) AS total_xga,
    (home_pts + away_pts) AS total_pts,
    (home_p + away_p) AS total_p,
    RANK() OVER (
        PARTITION BY season_end_year, league
        ORDER BY (home_pts + away_pts) DESC, (home_g + away_g - home_ga - away_ga) DESC, team ASC
    ) AS finishing_position,
    CASE
        WHEN tier = prev_tier THEN 'none'
        WHEN tier > prev_tier THEN 'relegated'
        WHEN tier < prev_tier THEN 'promoted'
        ELSE 'none'
    END AS season_start_flag,
    CASE
        WHEN tier = next_tier THEN 'none'
        WHEN tier > next_tier THEN 'promoted'
        WHEN tier < next_tier THEN 'relegated'
        ELSE 'none'
    END AS season_end_flag
FROM combined_stats
WHERE team = 'Auxerre'
ORDER BY season_end_year, country, tier, finishing_position;""").fetch_df()